# Task 2.3 — Train All 5 Model Sizes

Train Tiny, Small, Medium, Large, and XL models for **1 full epoch** (101.7M tokens) using:
- **LR = 3e-3** (best from Task 2.2 LR sweep)
- Cosine LR schedule with 5% warmup
- Effective batch: 65,536 tokens (phys_batch=4, grad_accum=16)
- AdamW, weight_decay=0.1, betas=(0.9, 0.95)
- Gradient clipping at 1.0

**Estimated times on M4 Pro (20-core GPU):**

| Model  | tok/s  | Est. Time |
|--------|--------|-----------|
| Tiny   | 78,931 | ~24 min   |
| Small  | 34,515 | ~48 min   |
| Medium | 23,649 | ~1.2 hrs  |
| Large  | 9,733  | ~2.9 hrs  |
| XL     | 4,509  | ~6.3 hrs  |

In [ ]:
import os, sys, json
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), 'ML-Final-Project'))
if not os.path.exists('configs'):
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
from scripts.train import train_model
import numpy as np
import matplotlib.pyplot as plt

# Best LR from Task 2.2
BEST_LR = 3e-3

CONFIGS = [
    'configs/tiny.json',
    'configs/small.json',
    'configs/medium.json',
    'configs/large.json',
    'configs/xl.json',
]
print(f'Will train {len(CONFIGS)} models with LR={BEST_LR:.1e}')

## Train Each Model

Run the cell below to train all 5 models sequentially.
Checkpoints are saved to `outputs/checkpoints/<name>/checkpoint_final.pt`.
Results accumulate in `outputs/results/scaling_results.json`.

> **Tip:** You can also run this from the terminal (runs in the background):
> ```bash
> python3 scripts/train.py --config configs/tiny.json   --lr 3e-3
> python3 scripts/train.py --config configs/small.json  --lr 3e-3
> python3 scripts/train.py --config configs/medium.json --lr 3e-3
> python3 scripts/train.py --config configs/large.json  --lr 3e-3
> python3 scripts/train.py --config configs/xl.json     --lr 3e-3
> ```

In [ ]:
all_results = []

for cfg_path in CONFIGS:
    with open(cfg_path) as f:
        name = json.load(f).get('name', cfg_path)

    # Skip if already trained (checkpoint exists)
    ckpt = f'outputs/checkpoints/{name}/checkpoint_final.pt'
    if os.path.exists(ckpt):
        print(f'[{name}] Checkpoint found — loading results from log.')
        import glob
        logs = glob.glob(f'outputs/logs/{name}_lr*.json')
        if logs:
            with open(sorted(logs)[0]) as f:
                r = json.load(f)
            all_results.append(r)
        continue

    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')
    r = train_model(
        config_path=cfg_path,
        lr=BEST_LR,
        data_dir='data/tokenized',
        output_dir='outputs',
        eval_interval=100,
        save_checkpoint=True,
    )
    all_results.append(r)
    print(f'Done: val_loss={r["val_loss"]:.4f}')

print('\nAll models complete.')

## Training Statistics Table (Task 2.5)

In [ ]:
# Load from scaling_results.json (authoritative source)
res_path = 'outputs/results/scaling_results.json'
if os.path.exists(res_path):
    with open(res_path) as f:
        data = json.load(f)

    print(f"{'Name':<10} {'Params':>10} {'d_model':>8} {'n_lay':>6} "
          f"{'n_hd':>5} {'d_ff':>6} {'ValLoss':>8} {'Time(h)':>8} "
          f"{'tok/s':>8} {'Mem(MB)':>8}")
    print('-' * 85)
    for r in data:
        t = r.get('total_time_seconds', 0)
        print(f"{r['name']:<10} {r['param_count']:>10,} {r['d_model']:>8} "
              f"{r['n_layers']:>6} {r['n_heads']:>5} {r['d_ff']:>6} "
              f"{r['val_loss']:>8.4f} {t/3600:>8.2f} "
              f"{r.get('tokens_per_second',0):>8,.0f} "
              f"{r.get('peak_memory_mb',0):>8.0f}")
else:
    print('scaling_results.json not found yet — train models first.')

## Training Loss Curves — All Models

In [ ]:
import glob

model_names = ['tiny', 'small', 'medium', 'large', 'xl']
colors = plt.cm.tab10(range(len(model_names)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
found = False

for name, color in zip(model_names, colors):
    # Find the full-epoch log (no max_steps restriction)
    logs = glob.glob(f'outputs/logs/{name}_lr*.json')
    if not logs: continue
    # Pick the one with lowest val_loss (= full epoch run)
    best = min(logs, key=lambda p: json.load(open(p)).get('val_loss', 1e9))
    with open(best) as f:
        log = json.load(f)
    entries = log.get('train_losses', [])
    if not entries: continue
    found = True
    steps  = [e['step']       for e in entries]
    t_loss = [e['train_loss'] for e in entries]
    v_loss = [e['val_loss']   for e in entries]
    axes[0].plot(steps, t_loss, label=name, color=color, linewidth=1.8)
    axes[1].plot(steps, v_loss, label=name, color=color, linewidth=1.8)

if found:
    for ax, title in zip(axes, ['Train Loss', 'Val Loss (sampled)']):
        ax.set_xlabel('Step', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title(f'{title} — All Models', fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle('Task 2.3 — Training Curves (LR=3e-3, 1 epoch, 101.7M tokens)', fontsize=12)
    fig.tight_layout()
    fig.savefig('outputs/plots/training_curves.png', dpi=150)
    plt.show()
    print('Saved: outputs/plots/training_curves.png')
else:
    print('No full-epoch log files found yet — train models first.')